In [0]:
import importlib
from config.config import *

In [0]:
import pycountry

# ISO-2 -> ISO-3 mapping
iso2_to_iso3 = {
    country.alpha_2: country.alpha_3
    for country in pycountry.countries
}

# Eurostat exceptions
iso2_to_iso3["EL"] = "GRC"
iso2_to_iso3["UK"] = "GBR"
iso2_to_iso3["XK"] = "XKX"

# UN M49 numeric code -> ISO-3
m49_to_iso3 = {
    int(country.numeric): country.alpha_3
    for country in pycountry.countries
}
# UN Exceptions
m49_to_iso3.update({
    5351: "BES",
    5352: "BES",
    5353: "BES"
})

In [0]:
from pyspark.sql import functions as F

iso2_mapping_df = spark.createDataFrame(
    [(k, v) for k, v in iso2_to_iso3.items()],
    ["geo_code", "country_iso3"]
)

m49_mapping_df = spark.createDataFrame(
    [(k, v) for k, v in m49_to_iso3.items()],
    ["area_code", "country_iso3"]
)

In [0]:
eurostat_tables = [
    "estat_tour_occ_arm",
    "estat_tour_occ_mnor",
    "estat_tour_occ_nim"
]

for table in eurostat_tables:

    df = spark.read.format("delta").load(
        f"{silver_path}/{table}"
    )

    if "country_iso3" in df.columns:
        df = df.drop("country_iso3")

    df = df.join(
        iso2_mapping_df,
        on="geo_code",
        how="left"
    )

    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{silver_path}/{table}")

    print(f"Normalized: {table}")

In [0]:
df = spark.read.format("delta").load(
    f"{silver_path}/un_tourism_accommodation_hotels"
)

if "reporter_country_iso3" in df.columns:
    df = df.drop("reporter_country_iso3")

reporter_mapping = m49_mapping_df.withColumnRenamed(
    "area_code", "reporter_area_code"
).withColumnRenamed(
    "country_iso3", "reporter_country_iso3"
)

df = df.join(
    reporter_mapping,
    on="reporter_area_code",
    how="left"
)

df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).save(f"{silver_path}/un_tourism_accommodation_hotels")

print("Normalized: un_tourism_accommodation_hotels")

In [0]:
df = spark.read.format("delta").load(
    f"{silver_path}/un_tourism_inbound_arrivals"
)

for col in ["reporter_country_iso3", "partner_country_iso3"]:
    if col in df.columns:
        df = df.drop(col)

reporter_mapping = m49_mapping_df.withColumnRenamed(
    "area_code", "reporter_area_code"
).withColumnRenamed(
    "country_iso3", "reporter_country_iso3"
)

partner_mapping = m49_mapping_df.withColumnRenamed(
    "area_code", "partner_area_code"
).withColumnRenamed(
    "country_iso3", "partner_country_iso3"
)

df = (
    df.join(reporter_mapping, on="reporter_area_code", how="left")
      .join(partner_mapping, on="partner_area_code", how="left")
)

df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).save(f"{silver_path}/un_tourism_inbound_arrivals")

print("Normalized: un_tourism_inbound_arrivals")

In [0]:
df = spark.read.format("delta").load(
    f"{silver_path}/world_bank_international_arrivals"
)

valid_iso3 = [
    country.alpha_3
    for country in pycountry.countries
]

df = df.withColumn(
    "country_iso3",
    F.when(
        F.col("country_code").isin(valid_iso3),
        F.col("country_code")
    )
)

df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).save(f"{silver_path}/world_bank_international_arrivals")

print("Normalized: world_bank_international_arrivals")

In [0]:
for table in [
    "estat_tour_occ_arm",
    "estat_tour_occ_mnor",
    "estat_tour_occ_nim",
    "un_tourism_accommodation_hotels",
    "un_tourism_inbound_arrivals",
    "world_bank_international_arrivals"
]:

    df = spark.read.format("delta").load(
        f"{silver_path}/{table}"
    )

    print(f"\n{'=' * 60}")
    print(table)
    print("=" * 60)

    cols = [
        c for c in df.columns
        if "country_iso3" in c
    ]

    for col in cols:
        df.select(col).distinct().orderBy(col).show(
            100, truncate=False
        )